In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
training_data = pd.read_csv("/kaggle/input/titanic/train.csv")
training_data.head()

In [ ]:
training_data.info()
training_data.describe()

In [ ]:
testing_data = pd.read_csv("/kaggle/input/titanic/test.csv")
testing_data.head()

In [ ]:
testing_data.info()
testing_data.describe()

In [ ]:
women = training_data.loc[training_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

In [ ]:
men = training_data.loc[training_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

In [ ]:
training_data["Age"] = training_data["Age"].fillna(training_data["Age"].median())
training_data["Embarked"] = training_data["Embarked"].fillna(training_data["Embarked"].mode()[0])
training_data["Fare"] = training_data["Fare"].fillna(training_data["Fare"].median())

testing_data["Age"] = testing_data["Age"].fillna(testing_data["Age"].median())
testing_data["Fare"] = testing_data["Fare"].fillna(testing_data["Fare"].median())
testing_data["Embarked"] = testing_data["Embarked"].fillna(testing_data["Embarked"].mode()[0])

In [ ]:
y = training_data["Survived"]

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = pd.get_dummies(training_data[features])
X_test = pd.get_dummies(testing_data[features])

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import numpy as np

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=7, random_state=1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=1)
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

In [ ]:
best_model = RandomForestClassifier(n_estimators=200, max_depth=7, random_state=1)
best_model.fit(X, y)
predictions = best_model.predict(X_test)

output = pd.DataFrame({'PassengerId': testing_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")